# Filter Litigants

Filters `01_cleaned_cases.csv` to retain only cases where at least one litigant is an individual person (not an organization), using `jieba.posseg` POS tagging on `case_name`.

- **`individual`**: at least one token in `case_name` tagged `nr` (人名)
- **`org`**: no `nr` tag AND name contains a known org keyword (公司, 协会, 中心, …)
- **`ambiguous`**: neither condition met — kept as individual for safety

Outputs:
- `01_filtered_cases.csv` — individual + ambiguous cases
- `01_discarded_cases.csv` — org-only cases for spot-checking

# Imports & Load Data

In [108]:
import pandas as pd
import jieba.posseg as pseg
import jieba
import os
jieba.load_userdict('additional_names.txt')

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [109]:
df = pd.read_csv("../data/00_raw_cases.csv")
print(f"Loaded {len(df)} cases")
df.head()

Loaded 81 cases


,case_name,case_type,court,date,case_href,full_text
0,北京丰冠体育有限公司等与中国滑雪协会等合同纠纷二审民事判决书,民事二审,北京市高级人民法院,2021-12-09,https://wenshu.court.gov.cn/website/wenshu/181...,北京市高级人民法院\n民 事 判 决 书\n（2021）京民终905号\n上诉人（原审被告）...
1,刘慷、吉林省射击射箭运动管理中心人事争议民事再审民事裁定书,民事再审,吉林省高级人民法院,2021-11-05,https://wenshu.court.gov.cn/website/wenshu/181...,吉林省高级人民法院\n民 事 裁 定 书\n（2021）吉民再270号\n再审申请人（一审原...
2,刘小宝、赵效男等买卖合同纠纷民事申请再审审查民事裁定书,民事再审,山东省高级人民法院,2021-07-05,https://wenshu.court.gov.cn/website/wenshu/181...,山东省高级人民法院\n民 事 裁 定 书\n（2021）鲁民申3670号\n再审申请人（一审...
3,马晓三、云南省人民检察院劳动争议再审民事判决书,民事再审,云南省高级人民法院,2020-06-09,https://wenshu.court.gov.cn/website/wenshu/181...,云南省高级人民法院\n民 事 判 决 书\n（2019）云民再30号\n抗诉机关：云南省人民...
4,李素萍与北京市木樨园体育运动技术学校劳动争议再审审查与审判监督民事裁定书,民事审判监督,北京市高级人民法院,2020-06-05,https://wenshu.court.gov.cn/website/wenshu/181...,北京市高级人民法院\n民 事 裁 定 书\n（2020）京民申2236号\n再审申请人（一审...


# Classify Litigant Type

In [112]:
ORG_KEYWORDS = {'公司', '协会', '中心', '委员会', '总局', '学院', '基金', '俱乐部', '队', '部', '体育'}

def classify_litigant(name):
    word_list = pseg.lcut(name)

    has_person = False
    for i, pair in enumerate(word_list):
        # Suppress nr tokens that are part of an org name
        if pair.flag == 'nr':
            if i + 1 < len(word_list):
                next_word = word_list[i + 1].word
                if any(kw in next_word for kw in ORG_KEYWORDS):
                    continue
            has_person = True
            break

    if has_person:
        return 'individual'
    has_org_keyword = any(kw in name for kw in ORG_KEYWORDS)
    if has_org_keyword:
        return 'org'
    return 'ambiguous'

In [114]:
df['litigant_type'] = df['case_name'].apply(classify_litigant)


# summary
df.groupby("litigant_type")["case_name"].count()

print(f"\nTotal: {len(df)}")
print(f"  → keeping (individual + ambiguous): {(df['litigant_type'] != 'org').sum()}")
print(f"  → discarding (org): {(df['litigant_type'] == 'org').sum()}")

litigant_type
individual    79
org            2
Name: case_name, dtype: int64


Total: 81
  → keeping (individual + ambiguous): 79
  → discarding (org): 2


# Spot-check Classifications

In [107]:
print("Sample individual cases:")
df[df['litigant_type'] == 'individual'][['case_name', 'litigant_type']].head(5)

print("\nSample org cases:")
df[df['litigant_type'] == 'org'][['case_name', 'litigant_type']].head(5)

print("\nSample ambiguous cases:")
df[df['litigant_type'] == 'ambiguous'][['case_name', 'litigant_type']].head(5)

Sample individual cases:


,case_name,litigant_type
1,刘慷、吉林省射击射箭运动管理中心人事争议民事再审民事裁定书,individual
2,刘小宝、赵效男等买卖合同纠纷民事申请再审审查民事裁定书,individual
3,马晓三、云南省人民检察院劳动争议再审民事判决书,individual
4,李素萍与北京市木樨园体育运动技术学校劳动争议再审审查与审判监督民事裁定书,individual
5,马俊岭等十三人故意伤害二审刑事附带民事判决书,individual



Sample org cases:


,case_name,litigant_type
0,北京丰冠体育有限公司等与中国滑雪协会等合同纠纷二审民事判决书,org
38,山东ＸＸ体育器材有限公司、山西ＸＸ体育文化发展有限公司民事一审民事判决书,org
77,原告纪程诉通化体育运动学校劳动争议一案一审民事判决书,org



Sample ambiguous cases:


,case_name,litigant_type


# Export

In [87]:
df_filtered  = df[df['litigant_type'] != 'org'].copy()
df_discarded = df[df['litigant_type'] == 'org'].copy()

df_filtered.to_csv("../data/01_filtered_cases.csv", index=False)
df_discarded.to_csv("../data/01_discarded_cases.csv", index=False)

print(f"01_filtered_cases.csv  → {len(df_filtered)} rows")
print(f"01_discarded_cases.csv → {len(df_discarded)} rows")
print(f"Saved to {os.path.abspath('../data/')}")

01_filtered_cases.csv  → 81 rows
01_discarded_cases.csv → 0 rows
Saved to /Users/wyx/Library/CloudStorage/OneDrive-DartmouthCollege/qss20_athlete_court_complaints/data


In [57]:
df.iloc[77][["case_name", "full_text"]]

case_name                           原告纪程诉通化体育运动学校劳动争议一案一审民事判决书
full_text    吉林省通化市东昌区人民法院\n民 事 判 决 书\n（2015）东民二初字第34号\n原告纪...
Name: 77, dtype: object